**ANNEXE CPU — exemple local de secours, distinct des compétitions Kaggle.**

# AML_01 — Du notebook au laboratoire hybride

**Question : Peut-on reproduire la même expérience en local et dans un environnement isolé ?**

Illustration courte ; ce fichier ne définit pas une journée supplémentaire. Équipes de trois, rôles tournants : données/validation, modèle/expériences, analyse/livraison.

**Parcours étudiant.** Les exemples fonctionnent ; les exercices demandent une décision et des preuves. Ne pas modifier les données ou splits pour améliorer artificiellement un score.

## Notions à comprendre

Une **observation** est une image 8x8 ; la cible est le chiffre 0 à 9. Les 64 pixels sont les entrées. Nous utilisons la log-loss multiclasse (à minimiser) et l’accuracy comme lecture complémentaire. La log-loss pénalise fortement une prédiction confiante mais fausse. Le classifieur naïf prédit les fréquences des classes apprises sur le train.

Les splits du benchmark sont figés : train pour apprendre, validation pour choisir, test sans labels pour la soumission finale. Les scripteurs ne sont pas identifiés : ce split stratifié ne prouve pas une généralisation à des personnes inconnues.

Le laboratoire garde **hypothèse + configuration + empreintes de données/code + métriques + prédictions + durée**. Une seed aide à reproduire ; elle ne garantit pas l’identité de résultats entre tout matériel et toutes versions. L’export Kaggle contient uniquement le code et les données autorisés, pas de secrets ni de labels test.

In [ ]:
from pathlib import Path
import sys, os, json, time, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
HERE=Path.cwd()
ROOT=next((p if (p/'mllab/core.py').exists() else p/'student') for p in [HERE,*HERE.parents] if (p/'mllab/core.py').exists() or (p/'student/mllab/core.py').exists())
sys.path.insert(0,str(ROOT))
from mllab.core import read_data, FEATURES, estimator, load_config, run, compare, evaluate
train=read_data('train'); valid=read_data('valid')
plt.rcParams.update({'figure.figsize':(10,4),'axes.spines.top':False,'axes.spines.right':False})
print('Racine du laboratoire :', ROOT)
print('Jeu de travail : chiffres manuscrits 8x8, benchmark pédagogique ; aucune compétition active choisie.')


## 1. Observer le contrat de données

In [ ]:
display(train.head())
assert not set(train.sample_id)&set(valid.sample_id)
fig,axes=plt.subplots(1,6,figsize=(12,2))
for ax,(_,row) in zip(axes,train.head(6).iterrows()):
 ax.imshow(row[FEATURES].to_numpy(dtype=float).reshape(8,8),cmap='gray');ax.set_title(f'Cible {int(row.target)}');ax.axis('off')
plt.show()
print('Test sans labels :', 'target' not in read_data('test'))

## 2. Une baseline puis une hypothèse

In [ ]:
cfg=load_config(ROOT/'configs/baseline.json')
base_path,base_score=run(cfg)
linear_cfg=load_config(ROOT/'configs/linear.json')
linear_path,linear_score=run(linear_cfg)
display(pd.DataFrame([{'modèle':'prior',**base_score},{'modèle':'logistique',**linear_score}]))
assert base_score['log_loss']>0 and linear_score['log_loss']>0

## 3. Préparer la boucle Kaggle

In [ ]:
from mllab.core import export_kaggle,pack_run,import_run
exchange=ROOT/'runs/exchange';exchange.mkdir(parents=True,exist_ok=True)
packet=export_kaggle(ROOT/'configs/linear.json',exchange/'experiment.zip')
result=pack_run(linear_path,exchange/'result_local.zip')
received=import_run(result)
print('Paquet à joindre comme donnée Kaggle :',packet)
print('Import vérifié localement :',received)
print('Ce test vérifie le format, pas une exécution sur les serveurs Kaggle.')

## 4. Comparer des preuves

In [ ]:
display(compare()[['name','model','log_loss','accuracy','seconds','data_hash']])
manifest=json.loads((linear_path/'manifest.json').read_text());display(manifest)
print('Avant comparaison : vérifier même split, mêmes classes et même métrique.')

## Atelier autonome et livrables

1. Écrire le QUID et expliquer pourquoi la log-loss et l’accuracy peuvent raconter des choses différentes.
2. Modifier C dans configs/linear.json en conservant une copie de la configuration initiale ; annoncer une hypothèse avant le run.
3. Faire exporter le paquet par un membre et importer les résultats par un autre. Sur Kaggle, suivre templates/KAGGLE_RUNNER.ipynb ; ne pas soumettre automatiquement.

**Journal :** hypothèse avant calcul, configuration, split, score, durée, mémoire si mesurée, décision et prochain essai. Pas de score ni de temps inventé. Les jeux du cours ne sont pas une compétition active.

In [ ]:
# Votre espace d’expérience : conserver un budget court, puis expliquer le résultat.
EXPERIMENT_HYPOTHESIS = 'À compléter avant de lancer un nouvel entraînement'
CONCLUSION = 'À compléter après comparaison'
print(EXPERIMENT_HYPOTHESIS)

## Questions de sortie

- Une accuracy de 90 % impose-t-elle une log-loss unique ?
- Quelle information permet de retrouver exactement les données utilisées ?

**Avant de quitter :** montrer le livrable, nommer une limite, annoncer le prochain essai. Chaque membre doit pouvoir expliquer une cellule.